In [1]:
%%capture output
!pip install -U huggingface_hub
from huggingface_hub import hf_hub_download
from huggingface_hub import login, HfApi, notebook_login
import copy
import os
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from PIL import Image, UnidentifiedImageError
from timm.models.layers import DropPath, trunc_normal_
from torch.utils.data import (DataLoader, Dataset, Subset,
                              WeightedRandomSampler)
from torchvision import datasets, transforms
from tqdm import tqdm

In [2]:
TRAIN_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/train' # Update if your path is different
TEST_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/test'
IMG_SIZE = 224
CLASSES = {
    'asian': 0, 'boho': 1, 'coastal': 2, 'contemporary': 3, 'craftsman': 4,
    'eclectic': 5, 'farmhouse': 6, 'french-country': 7, 'industrial': 8,
    'mediterranean': 9, 'minimalist': 10, 'modern': 11, 'scandinavian': 12,
    'shabby-chic-style': 13, 'southwestern': 14, 'tropical': 15, 'victorian': 16
}

In [3]:
def show_sample_images(directory):
    plt.figure(figsize=(15, 10))
    for i, style in enumerate(CLASSES.keys()):
        style_path = os.path.join(directory, style)
        img_name = os.listdir(style_path)[0]
        img_path = os.path.join(style_path, img_name)
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(4, 5, i+1)
        plt.imshow(img)
        plt.title(style)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# show_sample_images(TRAIN_DIR)

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), 
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_tf)
val_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=val_tf)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
indices = torch.randperm(len(train_dataset)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

train_targets = [train_subset.dataset.targets[i] for i in train_subset.indices]
class_counts = Counter(train_targets)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in train_targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_subset,
    batch_size=128,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=2)

# features, labels = next(iter(train_loader))
# print(f"Batch Shape: {features.shape}") 
# print(f"Labels Shape: {labels.shape}") # Use .shape, not .size (which is a method)
# print(f"Unique labels in this batch: {torch.unique(labels)}") # Check if we have variety

In [5]:
class LayerNorm(nn.Module):
    r""" LayerNorm that supports two data formats: channels_last (default) or channels_first. 
    The ordering of the dimensions in the inputs. channels_last corresponds to inputs with 
    shape (batch_size, height, width, channels) while channels_first corresponds to inputs 
    with shape (batch_size, channels, height, width).
    """
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ["channels_last", "channels_first"]:
            raise NotImplementedError 
        self.normalized_shape = (normalized_shape, )
    
    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x

In [6]:
class Block(nn.Module):
    r""" ConvNeXt Block. There are two equivalent implementations:
    (1) DwConv -> LayerNorm (channels_first) -> 1x1 Conv -> GELU -> 1x1 Conv; all in (N, C, H, W)
    (2) DwConv -> Permute to (N, H, W, C); LayerNorm (channels_last) -> Linear -> GELU -> Linear; Permute back
    We use (2) as we find it slightly faster in PyTorch
    
    Args:
        dim (int): Number of input channels.
        drop_path (float): Stochastic depth rate. Default: 0.0
        layer_scale_init_value (float): Init value for Layer Scale. Default: 1e-6.
    """
    def __init__(self, dim, drop_path=0., layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim) # depthwise conv
        self.norm = LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim) # pointwise/1x1 convs, implemented with linear layers
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones((dim)), 
                                    requires_grad=True) if layer_scale_init_value > 0 else None
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1) # (N, C, H, W) -> (N, H, W, C)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2) # (N, H, W, C) -> (N, C, H, W)

        x = input + self.drop_path(x)
        return x

In [7]:
class ConvNeXt(nn.Module):
    r""" ConvNeXt
        A PyTorch impl of : `A ConvNet for the 2020s`  -
          https://arxiv.org/pdf/2201.03545.pdf

    Args:
        in_chans (int): Number of input image channels. Default: 3
        num_classes (int): Number of classes for classification head. Default: 1000
        depths (tuple(int)): Number of blocks at each stage. Default: [3, 3, 9, 3]
        dims (int): Feature dimension at each stage. Default: [96, 192, 384, 768]
        drop_path_rate (float): Stochastic depth rate. Default: 0.
        layer_scale_init_value (float): Init value for Layer Scale. Default: 1e-6.
        head_init_scale (float): Init scaling value for classifier weights and biases. Default: 1.
    """
    def __init__(self, in_chans=3, num_classes=1000, 
                 depths=[3, 3, 9, 3], dims=[96, 192, 384, 768], drop_path_rate=0., 
                 layer_scale_init_value=1e-6, head_init_scale=1.,
                 ):
        super().__init__()

        self.downsample_layers = nn.ModuleList() # stem and 3 intermediate downsampling conv layers
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
            LayerNorm(dims[0], eps=1e-6, data_format="channels_first")
        )
        self.downsample_layers.append(stem)
        for i in range(3):
            downsample_layer = nn.Sequential(
                    LayerNorm(dims[i], eps=1e-6, data_format="channels_first"),
                    nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2),
            )
            self.downsample_layers.append(downsample_layer)

        self.stages = nn.ModuleList() # 4 feature resolution stages, each consisting of multiple residual blocks
        dp_rates=[x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))] 
        cur = 0
        for i in range(4):
            stage = nn.Sequential(
                *[Block(dim=dims[i], drop_path=dp_rates[cur + j], 
                layer_scale_init_value=layer_scale_init_value) for j in range(depths[i])]
            )
            self.stages.append(stage)
            cur += depths[i]

        self.norm = nn.LayerNorm(dims[-1], eps=1e-6) # final norm layer
        self.head = nn.Linear(dims[-1], num_classes)

        self.apply(self._init_weights)
        self.head.weight.data.mul_(head_init_scale)
        self.head.bias.data.mul_(head_init_scale)

    def _init_weights(self, m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            trunc_normal_(m.weight, std=.02)
            nn.init.constant_(m.bias, 0)

    def forward_features(self, x):
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        return self.norm(x.mean([-2, -1]))

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x

In [8]:
def convnext_tiny(num_classes=1000, drop_path_rate=0.1, **kwargs):
    """ConvNeXt-Tiny model"""
    return ConvNeXt(depths=[3, 3, 9, 3], dims=[96, 192, 384, 768], 
                    num_classes=num_classes, drop_path_rate=drop_path_rate, **kwargs)

def convnext_small(num_classes=1000, drop_path_rate=0.4, **kwargs):
    """ConvNeXt-Small model"""
    return ConvNeXt(depths=[3, 3, 27, 3], dims=[96, 192, 384, 768], 
                    num_classes=num_classes, drop_path_rate=drop_path_rate, **kwargs)

def convnext_base(num_classes=1000, drop_path_rate=0.5, **kwargs):
    """ConvNeXt-Base model"""
    return ConvNeXt(depths=[3, 3, 27, 3], dims=[128, 256, 512, 1024], 
                    num_classes=num_classes, drop_path_rate=drop_path_rate, **kwargs)

def convnext_large(num_classes=1000, drop_path_rate=0.5, **kwargs):
    """ConvNeXt-Large model"""
    return ConvNeXt(depths=[3, 3, 27, 3], dims=[192, 384, 768, 1536], 
                    num_classes=num_classes, drop_path_rate=drop_path_rate, **kwargs)

In [9]:
def get_pretrained_convnext(model_size="base", num_classes_target=17):
    configs = {
        "tiny":  {"depths": [3, 3, 9, 3],  "dims": [96, 192, 384, 768], 
                  "url": "https://dl.fbaipublicfiles.com/convnext/convnext_tiny_1k_224_ema.pth"},
        "small": {"depths": [3, 3, 27, 3], "dims": [96, 192, 384, 768], 
                  "url": "https://dl.fbaipublicfiles.com/convnext/convnext_small_1k_224_ema.pth"},
        "base":  {"depths": [3, 3, 27, 3], "dims": [128, 256, 512, 1024], 
                  "url": "https://dl.fbaipublicfiles.com/convnext/convnext_base_1k_224_ema.pth"},
    }
    
    cfg = configs[model_size]

    model = ConvNeXt(depths=cfg["depths"], dims=cfg["dims"], num_classes=1000)

    print(f"Downloading and loading {model_size} weights...")
    checkpoint = torch.hub.load_state_dict_from_url(cfg["url"], map_location="cpu", progress=True)
    
    if 'model' in checkpoint:
        state_dict = checkpoint['model']
    else:
        state_dict = checkpoint
        
    model.load_state_dict(state_dict)
    print("Weights loaded successfully.")

    dim_last = cfg["dims"][-1]
    model.head = nn.Linear(dim_last, num_classes_target)
    
    nn.init.trunc_normal_(model.head.weight, std=.02)
    nn.init.constant_(model.head.bias, 0)

    return model

# model = get_pretrained_convnext(model_size="base", num_classes_target=17)

In [10]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=10, save_path='best_convnext.pth'):
    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 10)

        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(train_loader, leave=True)
        
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad() 
            loss.backward()      
            optimizer.step()      

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            loop.set_description(f"Train Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)

        model.eval() 
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad(): 
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = correct / total
        
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
        print(f"Val Loss:   {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(epoch_val_loss)
        else:
            scheduler.step()
        
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), save_path)
            print(f"Creating checkpoint: Best Model Saved! ({best_acc:.4f})")

    model.load_state_dict(best_model_wts)
    return model, history

In [11]:
def train_full_data(model, train_loader, criterion, optimizer, scheduler, device, epochs=10, save_path='full_data_model.pth'):
    """
    Trains on ALL data. No validation. Saves the LAST checkpoint.
    """
    history = {'train_loss': [], 'train_acc': []}

    print(f"Training on 100% of data for {epochs} epochs...")

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(train_loader, leave=True)
        
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad() 
            loss.backward()      
            optimizer.step()      

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            loop.set_description(f"Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)

        print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(epoch_loss)
        else:
            scheduler.step()

    torch.save(model.state_dict(), save_path)
    print(f"Final model saved to {save_path}")

    return model, history

In [ ]:
full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_tf) 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

targets = full_dataset.targets
class_counts = Counter(targets)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

full_loader = DataLoader(
    full_dataset,
    batch_size=64,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

model = get_pretrained_convnext(model_size="base", num_classes_target=17)
model = model.to(device)

if torch.cuda.device_count() > 1:
    print(f"Success: Found {torch.cuda.device_count()} GPUs! Training in parallel.")
    model = nn.DataParallel(model)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

model, history = train_full_data(
    model, 
    full_loader, 
    criterion, 
    optimizer, 
    scheduler, 
    device, 
    epochs=10
)

In [ ]:
# notebook_login()
# repo_id = "mostafafaheem/ConvNeXT-base" 

# api = HfApi()

# api.create_repo(repo_id=repo_id, exist_ok=True)

# api.upload_file(
#     path_or_fileobj="/kaggle/working/full_data_model.pth",
#     path_in_repo="model.pth",
#     repo_id=repo_id,
#     repo_type="model"
# )
# print(f"Success! Model uploaded to: https://huggingface.co/{repo_id}")

In [14]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.file_names = sorted([
            f for f in os.listdir(root_dir) 
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        img_path = os.path.join(self.root_dir, fname)
        
        try:

            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Warning: Corrupted image found: {fname}. Using dummy input.")

            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='black')
        
        if self.transform:
            image = self.transform(image)        
            
        return image, fname

In [15]:
def load_model(num_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    dims = [128, 256, 512, 1024]
    depths = [3, 3, 27, 3]
    
    model = ConvNeXt(depths=depths, dims=dims, num_classes=num_classes)

    model_path = hf_hub_download(
        repo_id="mostafafaheem/my-convnext-model", 
        filename="model.pth",
        repo_type="model" 
    )
    

    state_dict = torch.load(model_path, map_location="cpu")
    
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace("module.", "") 
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict)
    
    model.to(device)
    model.eval()
    return model, device

In [16]:
def generate_predictions():
    test_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), # Simple resize for testing
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])
    
    test_dataset = TestDataset(TEST_DIR, transform=test_tf)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
    
    model, device = load_model(len(CLASSES))
    
    print(f"Starting inference on {len(test_dataset)} images...")
    
    all_filenames = []
    all_predictions = []
    
    with torch.no_grad():
        for images, filenames in tqdm(test_loader):
            images = images.to(device)
            
            # Forward pass
            outputs = model(images)
            
            _, preds = torch.max(outputs, 1)
            
            preds = preds.cpu().numpy()
            
            all_filenames.extend(filenames)
            all_predictions.extend(preds)
            
    return all_filenames, all_predictions

filenames, predictions = generate_predictions()

df = pd.DataFrame({
    'ImageName': filenames,    
    'ClassLabel': predictions 
})

df.to_csv('submission.csv', index=False)

model.pth:   0%|          | 0.00/350M [00:00<?, ?B/s]

Starting inference on 5482 images...


 13%|█▎        | 11/86 [00:10<00:50,  1.48it/s]

 43%|████▎     | 37/86 [00:27<00:32,  1.50it/s]

 44%|████▍     | 38/86 [00:28<00:31,  1.50it/s]

/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 60%|██████    | 52/86 [00:37<00:22,  1.48it/s]

100%|██████████| 86/86 [01:00<00:00,  1.42it/s]
